# DATASCI 185 · Quiz 4 Cheat Sheet

按老师的 **Practice Quiz 4 A–F** 排列。做题卡住时，直接找对应字母。这里用小例子演示写法，不用先做完整套复习。可从上到下运行，不需要额外数据文件。

| 题 | 看到什么就查这里 | 最重要的一行 |
|---|---|---|
| A | 随机拆表、算百分比 | `np.random.uniform(size=len(df))` |
| B | 按位置切片、降序 | `df.iloc[start:stop, col_start:col_stop]` |
| C | 单位转换、新列 | `df["column"].apply(function_name)` |
| D | 修改对象、限制上限 | `self.set_value(min(new_value, limit))` |
| E | 继承、覆盖方法、类变量 | `@classmethod` → `cls.attribute = value` |
| F | 按顺序配对 | 双层循环；符合条件就配对并 `break` |

这是按 **practice** 整理的，不代表正式考试范围。

## 起手式：读 CSV

`df = pd.read_csv("quiz_data/文件名.csv")`：如果 notebook 的工作目录是练习所在的文件夹，这条相对路径就从那里开始找。

`print(df.columns)` 看准确列名；`print(df.shape)` 看 (行数, 列数)；`len(df)` 只数行。

In [ ]:
import pandas as pd
import numpy as np

# 在 practice notebook 里，把下面文件名换成该题指定的 CSV：
# df = pd.read_csv("quiz_data/features.csv")

## A · 给每一行一个随机数，再用 query 分组

`uniform(size=n)` 生成 n 个 0 到 1 的随机数，每行一个。门槛为 0.4 时，落在左边的约占 40%，不是保证恰好 40%。分组条件写成 `<` 和 `>=`，门槛上的值也有归属。

`.query("列名 < @变量")`：表里的列名直接写；引号外定义的 Python 变量加 `@`。

**反引号（backticks）怎么写？** `Annual Wage(in USD)` 有空格和括号，在 `.query()` 里要用反引号包住列名；最外层还是双引号，`@` 则放在 Python 变量前。直接用 `df["列名"]` 取列时不加反引号。

In [ ]:
# 复习示例：约 25% 进 A，约 75% 进 B
sample = pd.DataFrame({"person_id": range(1000)})
n = len(sample)
cutoff = 0.25

sample["random_var"] = np.random.uniform(size=n)
part_a = sample.query("random_var < @cutoff")
part_b = sample.query("random_var >= @cutoff")

print("A:", round(len(part_a) / n * 100, 1), "%")
print("B:", round(len(part_b) / n * 100, 1), "%")
assert len(part_a) + len(part_b) == n

In [ ]:
# .query() 里的反引号：包住含空格或括号的列名
wages = pd.DataFrame({"Name": ["Ana", "Ben"], "Annual Wage(in USD)": [900, 1200]})
minimum = 1000
qualified = wages.query("`Annual Wage(in USD)` >= @minimum")
print(qualified)  # 只留下 Ben
assert qualified["Name"].tolist() == ["Ben"]

## B · `.iloc` / `.loc`：先切片，再排序

- **`.iloc[行位置, 列位置]`**：用方括号；逗号左边是行，右边是列。位置从 0 开始，切片不包含 stop。`:` 单独用表示选全部。
- **`.loc[行标签, 列标签]`**：看表上显示的标签，不是现在排第几。排序后，原来的行标签通常还在。
- **Practice B**：`df_subset = df_cars.iloc[28:33, 4:7]`。行位置 28–32 共 5 行，列位置 4–6 是 weight、acceleration、vehicle_id，共 3 列。可以先用 `print(df_cars.columns)` 确认列的位置。
- **排序**：`df_sorted = df_subset.sort_values(by="weight", ascending=False)`。按重量从高到低排，移动的是整行。先切再排，不等于先排再切。

In [ ]:
cars = pd.DataFrame({
    "model": ["A", "B", "C", "D"],
    "weight": [1300, 1700, 1200, 1600],
    "mpg": [32, 20, 35, 24],
})

# 取行位置 1、2 和列位置 1、2，再按重量降序
subset = cars.iloc[1:3, 1:3]
sorted_subset = subset.sort_values(by="weight", ascending=False)
print(sorted_subset)
assert sorted_subset.shape == (2, 2)

In [ ]:
# 接着上面的 cars 和 sorted_subset：
print(cars.iloc[0, 1])     # 第一行、第二列的一个值：1300
print(cars.iloc[:, 1])     # 所有行、第二列：整列 weight
print(cars.iloc[:2, :])    # 前两行、所有列
print(cars.loc[0, "weight"])  # 行标签 0、列名 weight

# sorted_subset 保留行标签 1 和 2：
print(sorted_subset.iloc[0, 0])    # 按位置取第一行、第一列：1700
print(sorted_subset.loc[2, "weight"])  # 按标签取第 2 行的 weight：1200
assert sorted_subset.iloc[0, 0] == 1700
assert sorted_subset.loc[2, "weight"] == 1200

## C · apply 转换一列，再用整列运算

`df["列名"].apply(fn)` 把这一列的值逐个交给 `fn`，收集返回值，得到 Series。这里传的是 **`fn`，不是 `fn()`**。新列用 `df["新列"] = 结果` 保存。

和你做过的欧元周薪题一样：转换用 `apply`，多个 Series 参与的公式直接算，不用写 for。平方写 `** 2`，不是 `^ 2`。

In [ ]:
# 示例：单价从分换成美元，再算每一行的总价
orders = pd.DataFrame({"Unit Price (cents)": [250, 125], "Quantity": [3, 4]})

def cents_to_dollars(cents):
    return cents / 100

unit_price_usd = orders["Unit Price (cents)"].apply(cents_to_dollars)
orders["Total (USD)"] = unit_price_usd * orders["Quantity"]
print(orders)
assert orders["Total (USD)"].tolist() == [7.5, 5.0]

## D · 实例方法：先检查，再改对象

`obj.method(参数)`：点前面的 `obj` 自动传给 `self`，括号里只放额外输入。方法要写在 class 里。

题目如果说「先验证、计数加一、增加属性但不超过上限、用 setter」，顺序就是：`assert` → 计数 → `min(原值 + 增量, 上限)` → 调 setter。

`min(103, 100)` 得到 100；`max` 在这里会选错。改变对象的方法通常不用 `return`。

In [ ]:
class Battery:
    def __init__(self, charge):
        self.charge = charge
        self.num_charges = 0

    def set_charge(self, value):
        assert 0 <= value <= 100, "charge must be between 0 and 100"
        self.charge = value

    def add_charge(self, amount):
        assert 0 <= amount <= 100, "invalid amount"
        self.num_charges += 1
        new_charge = min(self.charge + amount, 100)
        self.set_charge(new_charge)

battery = Battery(95)
battery.add_charge(8)
print(battery.charge, battery.num_charges)  # 100 1

# 捕获错误是为了让这份复习本可以 Run All。
try:
    battery.add_charge(-1)
except AssertionError as error:
    print("Expected:", error)

assert battery.num_charges == 1  # 无效输入没有改变计数

## E · 继承、override、class variable

- `class Child(Parent):` 类似 Java 的 `extends`。没重写 `__init__`，就沿用父类的。
- 子类写同名方法就是 override。需要改变父类的检查上限时，不要再调用那个只允许旧上限的父类 setter。
- 类变量直接写在 class 里面、方法外面。方法里用 `self.multiplier` 读取。要修改整类共用的值，用 `@classmethod` 和 `cls.multiplier = value`。写成 `self.multiplier = value` 只会给一个对象赋值。

In [ ]:
class ExtendedBattery(Battery):
    multiplier = 1.5

    def set_charge(self, value):
        assert 0 <= value <= 100 * self.multiplier, "charge out of range"
        self.charge = value

    @classmethod
    def set_multiplier(cls, value):
        assert 1 <= value <= 2, "multiplier must be between 1 and 2"
        cls.multiplier = value

first = ExtendedBattery(90)
second = ExtendedBattery(80)
first.set_charge(140)
print(first.charge)  # 140

ExtendedBattery.set_multiplier(1.8)
print(first.multiplier, second.multiplier)  # 1.8 1.8
assert first.multiplier == second.multiplier == 1.8

## F · 对象列表 + 找第一个合适的搭档

和 Assignment 6 一样，搭档属性存的是**另一个对象**，没有搭档时是 `None`。

1. 构造器：`self.列表 = []`。遍历字典时，循环变量是 key（名字）；`roster[name]` 是分数。
2. 外层：按顺序挑一个还没搭档的人。
3. 内层：跳过自己和已经配对的人；用 `abs(a - b) <= 上限` 测距离。
4. 找到就调用设置双方的方法，然后 `break`，只结束**内层**循环。

“第一个符合条件”不是“分数最接近”。下面 A 的分数是 50，先遇到 C 的 55，就和 C 配；即使后面 D 的 51 更接近，也不用继续找。

In [ ]:
class Member:
    def __init__(self, name, score):
        self.name = name
        self.score = score
        self.partner = None

    def pair_with(self, other):
        self.partner = other
        other.partner = self


class Team:
    max_gap = 5

    def __init__(self, roster):
        self.members = []
        for name in roster:
            self.members.append(Member(name, roster[name]))

    def pair_members(self):
        for member in self.members:
            if member.partner is not None:
                continue

            for other in self.members:
                if other is member or other.partner is not None:
                    continue

                if abs(member.score - other.score) <= self.max_gap:
                    member.pair_with(other)
                    break


team = Team({"A": 50, "B": 75, "C": 55, "D": 51})
team.pair_members()
for member in team.members:
    partner_name = None if member.partner is None else member.partner.name
    print(member.name, "→", partner_name)

assert team.members[0].partner is team.members[2]
assert team.members[2].partner is team.members[0]

## 交卷前 30 秒

- **A–C**：变量名和列名照题目；切片 stop 多写 1；降序是 `ascending=False`；`apply` 内不加调用括号。
- **D–E**：`assert` 放修改状态之前；边界包含时用 `<=`；检查是改一个对象还是改整个类。
- **F**：不能配自己；不能抢已有搭档；相差正好上限允许；找到第一位就停止内层循环。
- Practice 里写着 “should cause an error” 的格子，触发 `AssertionError` 正是预期。单独运行后继续下一格，不要为了让它不报错而改测试。